# Cleaning and Preprocessing

This notebook converts the audited Xeno-canto and eBird datasets into standardized, analysis-ready tables for downstream acoustic feature extraction, exploratory data analysis, clustering, dimensionality reduction, and machine-learning experiments.

The original audio recordings remain immutable. Cleaning in this notebook refers primarily to metadata standardization, validation, type conversion, derived-variable creation, and preservation of the quality-control decisions produced during the audit stage.

The final processed datasets will retain the audit flags associated with potentially challenging recordings so that these observations can be analyzed in later robustness and signal-enhancement experiments rather than being discarded.

## Preprocessing Philosophy

The preprocessing stage follows four principles:

1. **Preserve the original data.** Raw Xeno-canto and eBird files are not overwritten.
2. **Do not remove technically valid recordings solely because they are acoustically unusual.** The audit identified 275 technically valid recordings, including 67 recordings that were flagged for potentially challenging characteristics.
3. **Separate metadata cleaning from acoustic enhancement.** Metadata and analytical fields are standardized here; filtering, denoising, and other signal-processing experiments will be performed later without modifying the canonical recordings.
4. **Preserve provenance.** Audit actions, quality flags, source metadata, and derived variables remain traceable in the analysis-ready dataset.

In [1]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path.cwd().parents[2]

print("Project root:")
print(PROJECT_ROOT)

Project root:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity


In [3]:
SELECTED_METADATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "xenocanto_selected"
)

CLEANING_LOG_PATH = (
    PROJECT_ROOT
    / "data"
    / "manifests"
    / "cleaning_log.csv"
)

EBIRD_RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "ebird"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Selected metadata:", SELECTED_METADATA_DIR)
print("Cleaning log:", CLEANING_LOG_PATH)
print("eBird raw data:", EBIRD_RAW_DIR)
print("Processed data:", PROCESSED_DIR)

Selected metadata: c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\interim\xenocanto_selected
Cleaning log: c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\manifests\cleaning_log.csv
eBird raw data: c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\raw\ebird
Processed data: c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed


In [4]:
cleaning_log = pd.read_csv(
    CLEANING_LOG_PATH
)

print(
    "Cleaning log shape:",
    cleaning_log.shape
)

cleaning_log.head()

Cleaning log shape: (275, 16)


,recording_id,state,common_name,scientific_name,audit_action,audit_reason,flag_short_duration,flag_low_amplitude,flag_clipping,flag_low_quality,flag_any,file_size_bytes,duration_seconds,rms,peak_amplitude,quality
0,375492,CA,African Collared Dove,Streptopelia roseogrisea,KEEP,none,False,False,False,False,False,132062,5.251678,0.010715,0.071781,A
1,952203,CA,Allen's Hummingbird,Selasphorus sasin,KEEP,none,False,False,False,False,False,3709246,38.632562,0.011445,0.132172,A
2,452094,CA,American Coot,Fulica americana,KEEP,none,False,False,False,False,False,308179,18.960000,0.014746,0.704942,A
3,172887,CA,American Wigeon,Mareca americana,KEEP,none,False,False,False,False,False,187629,11.424000,0.014006,0.240556,B
4,449497,CA,Anna's Hummingbird,Calypte anna,KEEP,none,False,False,False,False,False,1768222,110.208000,0.024224,0.660087,A


In [5]:
print(
    cleaning_log["audit_action"].value_counts()
)

audit_action
KEEP    208
FLAG     67
Name: count, dtype: int64


In [6]:
selected_records = {}

for state in ["CA", "AZ", "TX"]:

    path = (
        SELECTED_METADATA_DIR
        / f"xc_a1_selected_{state}.json"
    )

    with open(path, encoding="utf-8") as f:
        selected_records[state] = json.load(f)

    print(
        f"{state}: {len(selected_records[state])} records"
    )

CA: 100 records
AZ: 75 records
TX: 100 records


In [7]:
metadata_rows = []

for state, records in selected_records.items():

    for record in records:

        metadata_rows.append({
            "state": state,
            "recording_id": str(record.get("id")),
            "common_name": record.get("en"),
            "scientific_name": (
                f"{record.get('gen', '')} "
                f"{record.get('sp', '')}"
            ).strip(),
            "quality": record.get("q"),
            "date": record.get("date"),
            "time": record.get("time"),
            "latitude": record.get("lat"),
            "longitude": record.get("lon"),
            "duration_metadata": record.get("length"),
            "audio_url": record.get("file"),
            "source_url": record.get("url"),
            "license": record.get("lic"),
        })

xc_metadata = pd.DataFrame(metadata_rows)

print("Rows:", len(xc_metadata))
print("Columns:", len(xc_metadata.columns))

xc_metadata.head()

Rows: 275
Columns: 13


,state,recording_id,common_name,scientific_name,quality,date,time,latitude,longitude,duration_metadata,audio_url,source_url,license
0,CA,375492,African Collared Dove,Streptopelia roseogrisea,A,2017-06-14,15:00,33.922,-117.2617,0:05,https://xeno-canto.org/375492/download,https://xeno-canto.org/375492,https://creativecommons.org/licenses/by-nc-sa/...
1,CA,952203,Allen's Hummingbird,Selasphorus sasin,A,2024-11-24,08:00,33.6104,-117.7392,0:38,https://xeno-canto.org/952203/download,https://xeno-canto.org/952203,https://creativecommons.org/licenses/by-nc-sa/...
2,CA,452094,American Coot,Fulica americana,A,2018-01-30,12:56,33.2016,-115.597,0:18,https://xeno-canto.org/452094/download,https://xeno-canto.org/452094,https://creativecommons.org/licenses/by-nc-sa/...
3,CA,172887,American Wigeon,Mareca americana,B,2012-03-23,09:54,32.564,-117.1256,0:11,https://xeno-canto.org/172887/download,https://xeno-canto.org/172887,https://creativecommons.org/licenses/by-nc-sa/...
4,CA,449497,Anna's Hummingbird,Calypte anna,A,2018-10-21,07:46,32.686,-117.243,1:50,https://xeno-canto.org/449497/download,https://xeno-canto.org/449497,https://creativecommons.org/licenses/by-nc-sa/...


In [8]:
# Standardize the merge keys before joining

xc_metadata["recording_id"] = (
    xc_metadata["recording_id"]
    .astype("string")
    .str.strip()
)

cleaning_log["recording_id"] = (
    cleaning_log["recording_id"]
    .astype("string")
    .str.strip()
)

xc_metadata["state"] = (
    xc_metadata["state"]
    .astype("string")
    .str.strip()
    .str.upper()
)

cleaning_log["state"] = (
    cleaning_log["state"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("Xeno-canto recording_id dtype:",
      xc_metadata["recording_id"].dtype)

print("Cleaning log recording_id dtype:",
      cleaning_log["recording_id"].dtype)

print("\nXeno-canto state dtype:",
      xc_metadata["state"].dtype)

print("Cleaning log state dtype:",
      cleaning_log["state"].dtype)

Xeno-canto recording_id dtype: string
Cleaning log recording_id dtype: string

Xeno-canto state dtype: string
Cleaning log state dtype: string


In [9]:
xc_keys = set(
    zip(
        xc_metadata["recording_id"],
        xc_metadata["state"]
    )
)

audit_keys = set(
    zip(
        cleaning_log["recording_id"],
        cleaning_log["state"]
    )
)

print("Xeno-canto keys:", len(xc_keys))
print("Audit keys:", len(audit_keys))

print(
    "Keys in Xeno-canto but not audit:",
    len(xc_keys - audit_keys)
)

print(
    "Keys in audit but not Xeno-canto:",
    len(audit_keys - xc_keys)
)

Xeno-canto keys: 275
Audit keys: 275
Keys in Xeno-canto but not audit: 0
Keys in audit but not Xeno-canto: 0


In [10]:
audit_columns = [
    "recording_id",
    "state",
    "audit_action",
    "audit_reason",
    "file_size_bytes",
    "duration_seconds",
    "rms",
    "peak_amplitude"
]

audit_subset = cleaning_log[audit_columns].copy()

master_metadata = xc_metadata.merge(
    audit_subset,
    on=["recording_id", "state"],
    how="left",
    validate="one_to_one"
)

print(
    "Master metadata shape:",
    master_metadata.shape
)

Master metadata shape: (275, 19)


In [11]:
print("Rows:", len(master_metadata))

print(
    "Missing audit actions:",
    master_metadata["audit_action"].isna().sum()
)

print(
    "Missing audit reasons:",
    master_metadata["audit_reason"].isna().sum()
)

print("\nAudit actions:")
print(
    master_metadata["audit_action"].value_counts(
        dropna=False
    )
)

Rows: 275
Missing audit actions: 0
Missing audit reasons: 0

Audit actions:
audit_action
KEEP    208
FLAG     67
Name: count, dtype: int64


In [12]:
print(cleaning_log.columns.tolist())

['recording_id', 'state', 'common_name', 'scientific_name', 'audit_action', 'audit_reason', 'flag_short_duration', 'flag_low_amplitude', 'flag_clipping', 'flag_low_quality', 'flag_any', 'file_size_bytes', 'duration_seconds', 'rms', 'peak_amplitude', 'quality']


In [13]:
print("Short duration:", cleaning_log["flag_short_duration"].sum())
print("Low amplitude:", cleaning_log["flag_low_amplitude"].sum())
print("Clipping:", cleaning_log["flag_clipping"].sum())
print("Low quality:", cleaning_log["flag_low_quality"].sum())
print("Any flag:", cleaning_log["flag_any"].sum())

Short duration: 8
Low amplitude: 53
Clipping: 8
Low quality: 0
Any flag: 67


In [14]:
audit_columns = [
    "recording_id",
    "state",
    "audit_action",
    "audit_reason",
    "flag_short_duration",
    "flag_low_amplitude",
    "flag_clipping",
    "flag_low_quality",
    "flag_any",
    "file_size_bytes",
    "duration_seconds",
    "rms",
    "peak_amplitude",
    "quality",
]

audit_subset = cleaning_log[audit_columns].copy()

master_metadata = xc_metadata.merge(
    audit_subset,
    on=["recording_id", "state"],
    how="left",
    validate="one_to_one"
)

print("Master metadata shape:", master_metadata.shape)

Master metadata shape: (275, 25)


In [15]:
print("Rows:", len(master_metadata))
print("Columns:", len(master_metadata.columns))

print("\nMissing audit actions:",
      master_metadata["audit_action"].isna().sum())

print("Missing audit reasons:",
      master_metadata["audit_reason"].isna().sum())

print("\nAudit actions:")
print(master_metadata["audit_action"].value_counts())

print("\nFlagged recordings:",
      master_metadata["flag_any"].sum())

Rows: 275
Columns: 25

Missing audit actions: 0
Missing audit reasons: 0

Audit actions:
audit_action
KEEP    208
FLAG     67
Name: count, dtype: int64

Flagged recordings: 67


In [16]:
print(master_metadata.dtypes)

state                  string[python]
recording_id           string[python]
common_name                    object
scientific_name                object
quality_x                      object
date                           object
time                           object
latitude                       object
longitude                      object
duration_metadata              object
audio_url                      object
source_url                     object
license                        object
audit_action                   object
audit_reason                   object
flag_short_duration              bool
flag_low_amplitude               bool
flag_clipping                    bool
flag_low_quality                 bool
flag_any                         bool
file_size_bytes                 int64
duration_seconds              float64
rms                           float64
peak_amplitude                float64
quality_y                      object
dtype: object


In [17]:
master_metadata.head(3)

,state,recording_id,common_name,scientific_name,quality_x,date,time,latitude,longitude,duration_metadata,...,flag_short_duration,flag_low_amplitude,flag_clipping,flag_low_quality,flag_any,file_size_bytes,duration_seconds,rms,peak_amplitude,quality_y
0,CA,375492,African Collared Dove,Streptopelia roseogrisea,A,2017-06-14,15:00,33.922,-117.2617,0:05,...,False,False,False,False,False,132062,5.251678,0.010715,0.071781,A
1,CA,952203,Allen's Hummingbird,Selasphorus sasin,A,2024-11-24,08:00,33.6104,-117.7392,0:38,...,False,False,False,False,False,3709246,38.632562,0.011445,0.132172,A
2,CA,452094,American Coot,Fulica americana,A,2018-01-30,12:56,33.2016,-115.597,0:18,...,False,False,False,False,False,308179,18.960000,0.014746,0.704942,A


In [18]:
print("quality_x == quality_y:",
      master_metadata["quality_x"].equals(
          master_metadata["quality_y"]
      ))

print("\nQuality values:")
print(
    pd.concat([
        master_metadata["quality_x"],
        master_metadata["quality_y"]
    ]).value_counts(dropna=False)
)

quality_x == quality_y: True

Quality values:
B    378
A    172
Name: count, dtype: int64


In [19]:
master_metadata["quality"] = master_metadata["quality_x"]

master_metadata = master_metadata.drop(
    columns=["quality_x", "quality_y"]
)

print("Columns after quality cleanup:")
print(master_metadata.columns.tolist())

print("\nQuality counts:")
print(
    master_metadata["quality"]
    .value_counts(dropna=False)
)

Columns after quality cleanup:
['state', 'recording_id', 'common_name', 'scientific_name', 'date', 'time', 'latitude', 'longitude', 'duration_metadata', 'audio_url', 'source_url', 'license', 'audit_action', 'audit_reason', 'flag_short_duration', 'flag_low_amplitude', 'flag_clipping', 'flag_low_quality', 'flag_any', 'file_size_bytes', 'duration_seconds', 'rms', 'peak_amplitude', 'quality']

Quality counts:
quality
B    189
A     86
Name: count, dtype: int64


In [20]:
# Text / identifier fields
text_columns = [
    "recording_id",
    "common_name",
    "scientific_name",
    "audio_url",
    "source_url",
    "license",
    "audit_action",
    "audit_reason",
    "quality",
]

for col in text_columns:
    master_metadata[col] = (
        master_metadata[col]
        .astype("string")
        .str.strip()
    )

# State
master_metadata["state"] = (
    master_metadata["state"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Numeric fields
numeric_columns = [
    "latitude",
    "longitude",
    "duration_seconds",
    "rms",
    "peak_amplitude",
    "file_size_bytes",
]

for col in numeric_columns:
    master_metadata[col] = pd.to_numeric(
        master_metadata[col],
        errors="coerce"
    )

print(master_metadata.dtypes)

state                  string[python]
recording_id           string[python]
common_name            string[python]
scientific_name        string[python]
date                           object
time                           object
latitude                      float64
longitude                     float64
duration_metadata              object
audio_url              string[python]
source_url             string[python]
license                string[python]
audit_action           string[python]
audit_reason           string[python]
flag_short_duration              bool
flag_low_amplitude               bool
flag_clipping                    bool
flag_low_quality                 bool
flag_any                         bool
file_size_bytes                 int64
duration_seconds              float64
rms                           float64
peak_amplitude                float64
quality                string[python]
dtype: object


In [21]:
print("Duration metadata examples:")
print(
    master_metadata["duration_metadata"]
    .value_counts(dropna=False)
    .head(15)
)

Duration metadata examples:
duration_metadata
0:08    9
0:17    9
0:05    8
0:07    8
0:06    8
0:18    8
0:24    8
0:16    8
0:13    6
0:45    6
0:11    6
0:21    6
0:15    6
0:14    6
0:12    5
Name: count, dtype: int64


In [22]:
print("Date examples:")
print(master_metadata["date"].head(10).tolist())

print("\nTime examples:")
print(master_metadata["time"].head(10).tolist())

Date examples:
['2017-06-14', '2024-11-24', '2018-01-30', '2012-03-23', '2018-10-21', '2023-06-02', '2017-07-18', '2021-04-26', '2025-01-16', '2012-02-16']

Time examples:
['15:00', '08:00', '12:56', '09:54', '07:46', '08:35', '18:00', '05:30', '14:31', '13:22']


In [23]:
# Convert recording date to datetime
master_metadata["date"] = pd.to_datetime(
    master_metadata["date"],
    format="%Y-%m-%d",
    errors="coerce"
)

print("Date dtype:", master_metadata["date"].dtype)
print("Missing dates:", master_metadata["date"].isna().sum())

Date dtype: datetime64[ns]
Missing dates: 1


In [24]:
missing_date = master_metadata[
    master_metadata["date"].isna()
]

print(
    missing_date[
        [
            "state",
            "recording_id",
            "common_name",
            "scientific_name",
            "date",
            "time"
        ]
    ].to_string(index=False)
)

state recording_id      common_name        scientific_name date time
   TX        70990 Green Kingfisher Chloroceryle americana  NaT    ?


In [25]:
xc_metadata.loc[
    xc_metadata["recording_id"] == "70990",
    [
        "recording_id",
        "state",
        "common_name",
        "scientific_name",
        "date",
        "time",
        "latitude",
        "longitude",
        "duration_metadata"
    ]
].T

,244
recording_id,70990
state,TX
common_name,Green Kingfisher
scientific_name,Chloroceryle americana
date,2010-04-00
time,?
latitude,29.8242
longitude,-99.5845
duration_metadata,0:11


In [26]:
# Preserve the original Xeno-canto temporal metadata
master_metadata["date_original"] = (
    master_metadata["date"]
    .astype("string")
)

master_metadata["time_original"] = (
    master_metadata["time"]
    .astype("string")
)

print(
    master_metadata.loc[
        master_metadata["recording_id"] == "70990",
        [
            "recording_id",
            "date_original",
            "time_original"
        ]
    ]
)

    recording_id date_original time_original
244        70990          <NA>             ?


In [27]:
original_temporal = xc_metadata[
    ["recording_id", "state", "date", "time"]
].copy()

original_temporal = original_temporal.rename(
    columns={
        "date": "date_original",
        "time": "time_original"
    }
)

master_metadata = master_metadata.drop(
    columns=["date_original", "time_original"],
    errors="ignore"
)

master_metadata = master_metadata.merge(
    original_temporal,
    on=["recording_id", "state"],
    how="left",
    validate="one_to_one"
)

print(
    master_metadata.loc[
        master_metadata["recording_id"] == "70990",
        [
            "recording_id",
            "date",
            "time",
            "date_original",
            "time_original"
        ]
    ]
)

    recording_id date time date_original time_original
244        70990  NaT    ?    2010-04-00             ?


In [28]:
print(
    master_metadata.loc[
        master_metadata["recording_id"] == "70990",
        [
            "recording_id",
            "date",
            "time",
            "date_original",
            "time_original"
        ]
    ]
)

    recording_id date time date_original time_original
244        70990  NaT    ?    2010-04-00             ?


In [29]:
master_metadata["datetime_local"] = pd.to_datetime(
    master_metadata["date"].astype("string")
    + " "
    + master_metadata["time"].astype("string"),
    errors="coerce"
)

print(
    master_metadata[
        [
            "state",
            "recording_id",
            "date",
            "time",
            "datetime_local"
        ]
    ].head(10)
)

print(
    "\nMissing local datetimes:",
    master_metadata["datetime_local"].isna().sum()
)

  state recording_id       date   time      datetime_local
0    CA       375492 2017-06-14  15:00 2017-06-14 15:00:00
1    CA       952203 2024-11-24  08:00 2024-11-24 08:00:00
2    CA       452094 2018-01-30  12:56 2018-01-30 12:56:00
3    CA       172887 2012-03-23  09:54 2012-03-23 09:54:00
4    CA       449497 2018-10-21  07:46 2018-10-21 07:46:00
5    CA       807038 2023-06-02  08:35 2023-06-02 08:35:00
6    CA       380363 2017-07-18  18:00 2017-07-18 18:00:00
7    CA       691650 2021-04-26  05:30 2021-04-26 05:30:00
8    CA       964245 2025-01-16  14:31 2025-01-16 14:31:00
9    CA       163349 2012-02-16  13:22 2012-02-16 13:22:00

Missing local datetimes: 7


In [30]:
print("Missing dates:",
      master_metadata["date"].isna().sum())

print("Missing times:",
      master_metadata["time"].isna().sum())

print("Missing local datetimes:",
      master_metadata["datetime_local"].isna().sum())

Missing dates: 1
Missing times: 0
Missing local datetimes: 7


In [31]:
temporal_issues = master_metadata[
    master_metadata["datetime_local"].isna()
][
    [
        "state",
        "recording_id",
        "common_name",
        "scientific_name",
        "date",
        "time",
        "date_original",
        "time_original"
    ]
]

print(
    temporal_issues.to_string(index=False)
)

state recording_id          common_name        scientific_name       date      time date_original time_original
   CA       132946 Calliope Hummingbird   Selasphorus calliope 2000-06-14         ?    2000-06-14             ?
   CA       120845        Virginia Rail        Rallus limicola 2012-04-18         ?    2012-04-18             ?
   AZ        48224 White-throated Swift   Aeronautes saxatalis 2009-11-05         ?    2009-11-05             ?
   TX       416763     Common Nighthawk       Chordeiles minor 2018-05-23  12:45:37    2018-05-23      12:45:37
   TX        56210     Downy Woodpecker    Dryobates pubescens 2010-06-17 17:20 CDT    2010-06-17     17:20 CDT
   TX        70990     Green Kingfisher Chloroceryle americana        NaT         ?    2010-04-00             ?
   TX         1382        Harris's Hawk   Parabuteo unicinctus 2001-12-25         ?    2001-12-25             ?


In [32]:
def parse_time(value):
    """
    Parse Xeno-canto time values while preserving local time.

    Supported examples:
    - 15:00
    - 12:45:37
    - 17:20 CDT
    - ?
    """
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    if value == "?":
        return pd.NaT

    # Remove an optional timezone abbreviation for the
    # local-clock time field. The timezone will be handled
    # separately.
    time_part = value.split()[0]

    parsed = pd.to_datetime(
        time_part,
        format="%H:%M:%S",
        errors="coerce"
    )

    if pd.isna(parsed):
        parsed = pd.to_datetime(
            time_part,
            format="%H:%M",
            errors="coerce"
        )

    if pd.isna(parsed):
        return pd.NaT

    return parsed.time()


master_metadata["time"] = (
    master_metadata["time_original"]
    .apply(parse_time)
)

print(
    master_metadata[
        [
            "recording_id",
            "time_original",
            "time"
        ]
    ].loc[
        master_metadata["recording_id"].isin(
            ["132946", "120845", "48224", "416763",
             "56210", "70990", "1382"]
        )
    ].to_string(index=False)
)

recording_id time_original     time
      132946             ?      NaT
      120845             ?      NaT
       48224             ?      NaT
      416763      12:45:37 12:45:37
       56210     17:20 CDT 17:20:00
       70990             ?      NaT
        1382             ?      NaT


In [33]:
print("Missing times:",
      master_metadata["time"].isna().sum())

Missing times: 5


In [34]:
def parse_date(value):
    """
    Parse Xeno-canto date values.

    Valid dates are converted to pandas datetime.
    Incomplete dates such as YYYY-MM-00 are treated
    as missing rather than being imputed.
    """
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    if value in ["", "?"]:
        return pd.NaT

    # Xeno-canto may use 00 for an unknown day.
    if value.endswith("-00"):
        return pd.NaT

    parsed = pd.to_datetime(
        value,
        format="%Y-%m-%d",
        errors="coerce"
    )

    return parsed


master_metadata["date"] = (
    master_metadata["date_original"]
    .apply(parse_date)
)

print("Missing dates:",
      master_metadata["date"].isna().sum())

print(
    master_metadata.loc[
        master_metadata["date"].isna(),
        [
            "recording_id",
            "date_original",
            "date"
        ]
    ].to_string(index=False)
)

Missing dates: 1
recording_id date_original date
       70990    2010-04-00  NaT


In [35]:
master_metadata["datetime_local"] = pd.to_datetime(
    master_metadata["date"].astype("string")
    + " "
    + master_metadata["time"].astype("string"),
    errors="coerce"
)

print(
    master_metadata[
        [
            "state",
            "recording_id",
            "date",
            "time",
            "datetime_local"
        ]
    ].head(10)
)

print(
    "\nMissing local datetimes:",
    master_metadata["datetime_local"].isna().sum()
)

  state recording_id       date      time      datetime_local
0    CA       375492 2017-06-14  15:00:00 2017-06-14 15:00:00
1    CA       952203 2024-11-24  08:00:00 2024-11-24 08:00:00
2    CA       452094 2018-01-30  12:56:00 2018-01-30 12:56:00
3    CA       172887 2012-03-23  09:54:00 2012-03-23 09:54:00
4    CA       449497 2018-10-21  07:46:00 2018-10-21 07:46:00
5    CA       807038 2023-06-02  08:35:00 2023-06-02 08:35:00
6    CA       380363 2017-07-18  18:00:00 2017-07-18 18:00:00
7    CA       691650 2021-04-26  05:30:00 2021-04-26 05:30:00
8    CA       964245 2025-01-16  14:31:00 2025-01-16 14:31:00
9    CA       163349 2012-02-16  13:22:00 2012-02-16 13:22:00

Missing local datetimes: 5


In [36]:
def extract_source_timezone(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    parts = value.split()

    if len(parts) >= 2:
        timezone = parts[-1].upper()

        if timezone in ["PST", "PDT", "MST", "MDT", "CST", "CDT"]:
            return timezone

    return pd.NA


master_metadata["source_timezone"] = (
    master_metadata["time_original"]
    .apply(extract_source_timezone)
    .astype("string")
)

print(
    master_metadata[
        master_metadata["source_timezone"].notna()
    ][
        [
            "recording_id",
            "state",
            "date",
            "time",
            "time_original",
            "source_timezone"
        ]
    ].to_string(index=False)
)

recording_id state       date     time time_original source_timezone
       56210    TX 2010-06-17 17:20:00     17:20 CDT             CDT


In [37]:
master_metadata["has_date"] = (
    master_metadata["date"].notna()
)

master_metadata["has_time"] = (
    master_metadata["time"].notna()
)

master_metadata["has_datetime_local"] = (
    master_metadata["datetime_local"].notna()
)

print(
    "Records with valid date:",
    master_metadata["has_date"].sum()
)

print(
    "Records with valid time:",
    master_metadata["has_time"].sum()
)

print(
    "Records with valid local datetime:",
    master_metadata["has_datetime_local"].sum()
)

print("\nTemporal availability:")
print(
    master_metadata[
        [
            "has_date",
            "has_time",
            "has_datetime_local"
        ]
    ].value_counts()
)

Records with valid date: 274
Records with valid time: 270
Records with valid local datetime: 270

Temporal availability:
has_date  has_time  has_datetime_local
True      True      True                  270
          False     False                   4
False     False     False                   1
Name: count, dtype: int64


In [38]:
# ---------------------------------------------------------
# Temporal features based on LOCAL recording time
# ---------------------------------------------------------

master_metadata["year"] = (
    master_metadata["datetime_local"].dt.year
)

master_metadata["month"] = (
    master_metadata["datetime_local"].dt.month
)

master_metadata["day_of_year"] = (
    master_metadata["datetime_local"].dt.dayofyear
)

master_metadata["hour"] = (
    master_metadata["datetime_local"].dt.hour
)

master_metadata["minute"] = (
    master_metadata["datetime_local"].dt.minute
)

print(
    master_metadata[
        [
            "recording_id",
            "state",
            "datetime_local",
            "year",
            "month",
            "day_of_year",
            "hour",
            "minute"
        ]
    ].head(10)
)

  recording_id state      datetime_local    year  month  day_of_year  hour  \
0       375492    CA 2017-06-14 15:00:00  2017.0    6.0        165.0  15.0   
1       952203    CA 2024-11-24 08:00:00  2024.0   11.0        329.0   8.0   
2       452094    CA 2018-01-30 12:56:00  2018.0    1.0         30.0  12.0   
3       172887    CA 2012-03-23 09:54:00  2012.0    3.0         83.0   9.0   
4       449497    CA 2018-10-21 07:46:00  2018.0   10.0        294.0   7.0   
5       807038    CA 2023-06-02 08:35:00  2023.0    6.0        153.0   8.0   
6       380363    CA 2017-07-18 18:00:00  2017.0    7.0        199.0  18.0   
7       691650    CA 2021-04-26 05:30:00  2021.0    4.0        116.0   5.0   
8       964245    CA 2025-01-16 14:31:00  2025.0    1.0         16.0  14.0   
9       163349    CA 2012-02-16 13:22:00  2012.0    2.0         47.0  13.0   

   minute  
0     0.0  
1     0.0  
2    56.0  
3    54.0  
4    46.0  
5    35.0  
6     0.0  
7    30.0  
8    31.0  
9    22.0  


In [39]:
def assign_season(month):
    if pd.isna(month):
        return pd.NA

    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"


master_metadata["season"] = (
    master_metadata["month"]
    .apply(assign_season)
    .astype("string")
)

print(
    master_metadata[
        ["month", "season"]
    ].drop_duplicates().sort_values("month")
)

    month  season
2     1.0  Winter
9     2.0  Winter
3     3.0  Spring
7     4.0  Spring
11    5.0  Spring
0     6.0  Summer
6     7.0  Summer
14    8.0  Summer
92    9.0    Fall
4    10.0    Fall
1    11.0    Fall
32   12.0  Winter
15    NaN    <NA>


In [40]:
print("Temporal feature missingness:")

print(
    master_metadata[
        [
            "year",
            "month",
            "day_of_year",
            "hour",
            "minute",
            "season"
        ]
    ].isna().sum()
)

print("\nSeason distribution:")
print(
    master_metadata["season"]
    .value_counts(dropna=False)
)

print("\nHour distribution:")
print(
    master_metadata["hour"]
    .value_counts(dropna=False)
    .sort_index()
)

Temporal feature missingness:
year           5
month          5
day_of_year    5
hour           5
minute         5
season         5
dtype: int64

Season distribution:
season
Spring    107
Summer     73
Winter     71
Fall       19
<NA>        5
Name: count, dtype: Int64

Hour distribution:
hour
0.0      1
3.0      2
4.0      3
5.0     12
6.0     15
7.0     47
8.0     25
9.0     24
10.0    22
11.0    11
12.0    15
13.0    12
14.0    12
15.0    13
16.0     6
17.0    17
18.0     8
19.0     9
20.0     6
21.0     8
22.0     1
23.0     1
NaN      5
Name: count, dtype: int64


In [41]:
def duration_to_seconds(value):
    """
    Convert Xeno-canto duration strings such as
    '0:08', '0:17', or '1:23' into seconds.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if not value or value == "?":
        return np.nan

    parts = value.split(":")

    try:
        if len(parts) == 2:
            minutes = int(parts[0])
            seconds = int(parts[1])

            if not 0 <= seconds < 60:
                return np.nan

            return minutes * 60 + seconds

        return np.nan

    except (ValueError, TypeError):
        return np.nan


master_metadata["duration_metadata_seconds"] = (
    master_metadata["duration_metadata"]
    .apply(duration_to_seconds)
)

print(
    master_metadata[
        [
            "recording_id",
            "duration_metadata",
            "duration_metadata_seconds",
            "duration_seconds"
        ]
    ].head(10)
)

  recording_id duration_metadata  duration_metadata_seconds  duration_seconds
0       375492              0:05                          5          5.251678
1       952203              0:38                         38         38.632562
2       452094              0:18                         18         18.960000
3       172887              0:11                         11         11.424000
4       449497              1:50                        110        110.208000
5       807038              0:25                         25         25.680000
6       380363              0:24                         24         24.663220
7       691650              0:00                          0          0.696000
8       964245              0:05                          5          5.166729
9       163349              0:03                          3          3.192000


In [42]:
print(
    "Missing metadata duration:",
    master_metadata["duration_metadata_seconds"].isna().sum()
)

print(
    "Missing decoded duration:",
    master_metadata["duration_seconds"].isna().sum()
)

Missing metadata duration: 0
Missing decoded duration: 0


In [43]:
master_metadata["duration_difference_seconds"] = (
    master_metadata["duration_seconds"]
    - master_metadata["duration_metadata_seconds"]
)

print(
    master_metadata[
        [
            "duration_metadata_seconds",
            "duration_seconds",
            "duration_difference_seconds"
        ]
    ].describe()
)

       duration_metadata_seconds  duration_seconds  \
count                 275.000000        275.000000   
mean                   45.585455         46.038436   
std                    64.853252         64.860825   
min                     0.000000          0.696000   
25%                    12.500000         12.864331   
50%                    24.000000         23.974739   
75%                    54.000000         54.105252   
max                   646.000000        646.992000   

       duration_difference_seconds  
count                   275.000000  
mean                      0.452982  
std                       0.300460  
min                      -0.067052  
25%                       0.184000  
50%                       0.458186  
75%                       0.708424  
max                       0.992000  


In [44]:
short_recordings = master_metadata[
    master_metadata["flag_short_duration"]
][
    [
        "recording_id",
        "state",
        "common_name",
        "duration_metadata",
        "duration_metadata_seconds",
        "duration_seconds",
        "flag_low_amplitude",
        "flag_clipping",
        "audit_action"
    ]
].sort_values("duration_seconds")

print(
    short_recordings.to_string(index=False)
)

recording_id state               common_name duration_metadata  duration_metadata_seconds  duration_seconds  flag_low_amplitude  flag_clipping audit_action
      691650    CA Black-chinned Hummingbird              0:00                          0          0.696000                True          False         FLAG
      288791    TX         Belted Kingfisher              0:00                          0          0.744000               False          False         FLAG
      845953    TX            Common Ostrich              0:00                          0          0.842086               False          False         FLAG
      862339    TX        Black-necked Stilt              0:01                          1          1.056000               False          False         FLAG
       75505    AZ        Anna's Hummingbird              0:01                          1          1.861995               False          False         FLAG
      656787    TX           Common Poorwill              0:02  

In [45]:
def classify_duration(seconds):
    if pd.isna(seconds):
        return pd.NA
    elif seconds < 3:
        return "Very short (<3s)"
    elif seconds < 10:
        return "Short (3–10s)"
    elif seconds < 30:
        return "Medium (10–30s)"
    else:
        return "Long (≥30s)"


master_metadata["duration_category"] = (
    master_metadata["duration_seconds"]
    .apply(classify_duration)
    .astype("string")
)

print(
    master_metadata["duration_category"]
    .value_counts(dropna=False)
)

duration_category
Long (≥30s)         119
Medium (10–30s)     103
Short (3–10s)        45
Very short (<3s)      8
Name: count, dtype: Int64


In [46]:
duration_flag_summary = (
    master_metadata
    .groupby("duration_category", dropna=False)
    .agg(
        recordings=("recording_id", "count"),
        flagged=("flag_any", "sum"),
        mean_duration=("duration_seconds", "mean")
    )
    .reset_index()
)

print(duration_flag_summary)

  duration_category  recordings  flagged  mean_duration
0       Long (≥30s)         119       31      87.627611
1   Medium (10–30s)         103       17      18.655789
2     Short (3–10s)          45       11       6.646450
3  Very short (<3s)           8        8       1.530963


In [47]:
species_audit = (
    master_metadata[
        ["recording_id", "state", "common_name", "scientific_name"]
    ]
    .copy()
)

print("Missing common names:",
      species_audit["common_name"].isna().sum())

print("Missing scientific names:",
      species_audit["scientific_name"].isna().sum())

print("Unique common names:",
      species_audit["common_name"].nunique())

print("Unique scientific names:",
      species_audit["scientific_name"].nunique())

Missing common names: 0
Missing scientific names: 0
Unique common names: 159
Unique scientific names: 159


In [48]:
common_to_scientific = (
    species_audit
    .groupby("common_name")["scientific_name"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    common_to_scientific[
        common_to_scientific > 1
    ]
)

Series([], Name: scientific_name, dtype: int64)


In [49]:
scientific_to_common = (
    species_audit
    .groupby("scientific_name")["common_name"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    scientific_to_common[
        scientific_to_common > 1
    ]
)

Series([], Name: common_name, dtype: int64)


## Species Label Audit

The selected Xeno-canto dataset contains complete species labels for all 275 recordings. No recordings are missing either a common name or scientific name, and 159 unique species are represented in the dataset.

A consistency check found no cases in which a single common name was associated with multiple scientific names or a single scientific name was associated with multiple common names. Therefore, no species-label corrections were required during preprocessing. The original Xeno-canto taxonomy fields are retained for downstream analysis.

In [50]:
master_metadata["species_id"] = (
    master_metadata["scientific_name"]
    .astype("string")
    .str.strip()
)

print(
    master_metadata[
        ["common_name", "scientific_name", "species_id"]
    ].head(10)
)

print(
    "Missing species IDs:",
    master_metadata["species_id"].isna().sum()
)

print(
    "Unique species IDs:",
    master_metadata["species_id"].nunique()
)

                 common_name           scientific_name  \
0      African Collared Dove  Streptopelia roseogrisea   
1        Allen's Hummingbird         Selasphorus sasin   
2              American Coot          Fulica americana   
3            American Wigeon          Mareca americana   
4         Anna's Hummingbird              Calypte anna   
5         Band-tailed Pigeon      Patagioenas fasciata   
6                 Black Rail    Laterallus jamaicensis   
7  Black-chinned Hummingbird     Archilochus alexandri   
8           Blue-winged Teal           Spatula discors   
9                Brant Goose           Branta bernicla   

                 species_id  
0  Streptopelia roseogrisea  
1         Selasphorus sasin  
2          Fulica americana  
3          Mareca americana  
4              Calypte anna  
5      Patagioenas fasciata  
6    Laterallus jamaicensis  
7     Archilochus alexandri  
8           Spatula discors  
9           Branta bernicla  
Missing species IDs: 0
Unique s

In [51]:
ebird_files = {
    "CA": EBIRD_RAW_DIR / "ebird_observations_raw_CA.json",
    "AZ": EBIRD_RAW_DIR / "ebird_observations_raw_AZ.json",
    "TX": EBIRD_RAW_DIR / "ebird_observations_raw_TX.json",
}

for state, path in ebird_files.items():
    print(f"\n{'=' * 60}")
    print(f"{state}: {path.name}")

    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    print("Records:", len(records))

    if records:
        print("Fields:")
        print(list(records[0].keys()))

        print("\nFirst record:")
        print(records[0])


CA: ebird_observations_raw_CA.json
Records: 495
Fields:
['speciesCode', 'comName', 'sciName', 'locId', 'locName', 'obsDt', 'howMany', 'lat', 'lng', 'obsValid', 'obsReviewed', 'locationPrivate', 'subId']

First record:
{'speciesCode': 'leasan', 'comName': 'Least Sandpiper', 'sciName': 'Calidris minutilla', 'locId': 'L509037', 'locName': 'Morro Bay--Audubon overlook', 'obsDt': '2026-09-19 10:02', 'howMany': 1, 'lat': 35.3316478, 'lng': -120.8383334, 'obsValid': True, 'obsReviewed': False, 'locationPrivate': False, 'subId': 'S394291646'}

AZ: ebird_observations_raw_AZ.json
Records: 373
Fields:
['speciesCode', 'comName', 'sciName', 'locId', 'locName', 'obsDt', 'howMany', 'lat', 'lng', 'obsValid', 'obsReviewed', 'locationPrivate', 'subId']

First record:
{'speciesCode': 'cubthr', 'comName': 'Curve-billed Thrasher', 'sciName': 'Toxostoma curvirostre', 'locId': 'L15663702', 'locName': '2436 West Bartlett Way, Queen Creek, Arizona, US (33.19, -111.592)', 'obsDt': '2026-09-19 09:50', 'howMany'

In [52]:
ebird_rows = []

for state, path in ebird_files.items():

    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    for record in records:
        ebird_rows.append({
            "state": state,
            "species_code": record.get("speciesCode"),
            "common_name": record.get("comName"),
            "scientific_name": record.get("sciName"),
            "location_id": record.get("locId"),
            "location_name": record.get("locName"),
            "observation_datetime": record.get("obsDt"),
            "how_many": record.get("howMany"),
            "latitude": record.get("lat"),
            "longitude": record.get("lng"),
            "observation_valid": record.get("obsValid"),
            "observation_reviewed": record.get("obsReviewed"),
            "location_private": record.get("locationPrivate"),
            "submission_id": record.get("subId"),
        })

ebird_metadata = pd.DataFrame(ebird_rows)

print("Shape:", ebird_metadata.shape)

print("\nColumns:")
print(ebird_metadata.columns.tolist())

print("\nRecords by state:")
print(ebird_metadata["state"].value_counts())

print("\nFirst 5 rows:")
display(ebird_metadata.head())

Shape: (1326, 14)

Columns:
['state', 'species_code', 'common_name', 'scientific_name', 'location_id', 'location_name', 'observation_datetime', 'how_many', 'latitude', 'longitude', 'observation_valid', 'observation_reviewed', 'location_private', 'submission_id']

Records by state:
state
CA    495
TX    458
AZ    373
Name: count, dtype: int64

First 5 rows:


,state,species_code,common_name,scientific_name,location_id,location_name,observation_datetime,how_many,latitude,longitude,observation_valid,observation_reviewed,location_private,submission_id
0,CA,leasan,Least Sandpiper,Calidris minutilla,L509037,Morro Bay--Audubon overlook,2026-09-19 10:02,1.0,35.331648,-120.838333,True,False,False,S394291646
1,CA,normoc,Northern Mockingbird,Mimus polyglottos,L509037,Morro Bay--Audubon overlook,2026-09-19 10:02,1.0,35.331648,-120.838333,True,False,False,S394291646
2,CA,calthr,California Thrasher,Toxostoma redivivum,L509037,Morro Bay--Audubon overlook,2026-09-19 10:02,1.0,35.331648,-120.838333,True,False,False,S394291646
3,CA,belkin1,Belted Kingfisher,Megaceryle alcyon,L53117291,"Arastradero Lake, Palo Alto US-CA 37.38182, -1...",2026-09-19 10:00,1.0,37.381815,-122.176246,True,False,True,S394290433
4,CA,oaktit,Oak Titmouse,Baeolophus inornatus,L28362392,"315 E Miramar Ave, Claremont US-CA 34.12918, -...",2026-09-19 09:59,1.0,34.129179,-117.710442,True,False,True,S394289935


In [53]:
ebird_text_columns = [
    "state",
    "species_code",
    "common_name",
    "scientific_name",
    "location_id",
    "location_name",
    "submission_id",
]

for col in ebird_text_columns:
    ebird_metadata[col] = (
        ebird_metadata[col]
        .astype("string")
        .str.strip()
    )

ebird_metadata["state"] = (
    ebird_metadata["state"]
    .str.upper()
)

ebird_numeric_columns = [
    "latitude",
    "longitude",
    "how_many",
]

for col in ebird_numeric_columns:
    ebird_metadata[col] = pd.to_numeric(
        ebird_metadata[col],
        errors="coerce"
    )

ebird_metadata["observation_valid"] = (
    ebird_metadata["observation_valid"]
    .astype("boolean")
)

ebird_metadata["observation_reviewed"] = (
    ebird_metadata["observation_reviewed"]
    .astype("boolean")
)

ebird_metadata["location_private"] = (
    ebird_metadata["location_private"]
    .astype("boolean")
)

In [54]:
ebird_metadata["observation_datetime_original"] = (
    ebird_metadata["observation_datetime"]
    .astype("string")
    .str.strip()
)

ebird_metadata["observation_datetime"] = pd.to_datetime(
    ebird_metadata["observation_datetime_original"],
    errors="coerce"
)

print(
    ebird_metadata[
        [
            "observation_datetime_original",
            "observation_datetime"
        ]
    ].head(10)
)

print(
    "\nMissing parsed observation datetimes:",
    ebird_metadata["observation_datetime"].isna().sum()
)

  observation_datetime_original observation_datetime
0              2026-09-19 10:02  2026-09-19 10:02:00
1              2026-09-19 10:02  2026-09-19 10:02:00
2              2026-09-19 10:02  2026-09-19 10:02:00
3              2026-09-19 10:00  2026-09-19 10:00:00
4              2026-09-19 09:59  2026-09-19 09:59:00
5              2026-09-19 09:57  2026-09-19 09:57:00
6              2026-09-19 09:57  2026-09-19 09:57:00
7              2026-09-19 09:57  2026-09-19 09:57:00
8              2026-09-19 09:57  2026-09-19 09:57:00
9              2026-09-19 09:57  2026-09-19 09:57:00

Missing parsed observation datetimes: 1


In [55]:
invalid_ebird_datetime = ebird_metadata[
    ebird_metadata["observation_datetime"].isna()
][
    [
        "state",
        "species_code",
        "common_name",
        "scientific_name",
        "location_id",
        "location_name",
        "observation_datetime_original",
        "how_many",
        "latitude",
        "longitude",
        "submission_id",
    ]
]

print(invalid_ebird_datetime.to_string(index=False))

state species_code      common_name scientific_name location_id                     location_name observation_datetime_original  how_many  latitude  longitude submission_id
   TX       sooshe Sooty Shearwater  Ardenna grisea   L77277315 Mkr 71, Mustang Island Gulf Beach                    2026-09-04       1.0 27.738247 -97.126701    S393510921


In [56]:
ebird_metadata["observation_date"] = pd.to_datetime(
    ebird_metadata["observation_datetime_original"]
    .str.extract(r"^(\d{4}-\d{2}-\d{2})")[0],
    format="%Y-%m-%d",
    errors="coerce"
)

print(
    "Missing observation dates:",
    ebird_metadata["observation_date"].isna().sum()
)

print(
    ebird_metadata[
        ebird_metadata["observation_date"].isna()
    ][
        [
            "observation_datetime_original",
            "observation_date"
        ]
    ]
)

Missing observation dates: 0
Empty DataFrame
Columns: [observation_datetime_original, observation_date]
Index: []


In [57]:
ebird_metadata["year"] = (
    ebird_metadata["observation_date"].dt.year
)

ebird_metadata["month"] = (
    ebird_metadata["observation_date"].dt.month
)

ebird_metadata["day_of_year"] = (
    ebird_metadata["observation_date"].dt.dayofyear
)

In [58]:
ebird_metadata["hour"] = (
    ebird_metadata["observation_datetime"].dt.hour
)

ebird_metadata["minute"] = (
    ebird_metadata["observation_datetime"].dt.minute
)

ebird_metadata["has_observation_time"] = (
    ebird_metadata["observation_datetime"].notna()
)

In [59]:
ebird_metadata["season"] = (
    ebird_metadata["month"]
    .apply(assign_season)
    .astype("string")
)

print(
    ebird_metadata[
        [
            "observation_date",
            "observation_datetime",
            "year",
            "month",
            "hour",
            "season",
            "has_observation_time"
        ]
    ].tail()
)

     observation_date observation_datetime  year  month  hour  season  \
1321       2026-08-21  2026-08-21 11:45:00  2026      8  11.0  Summer   
1322       2026-08-21  2026-08-21 09:45:00  2026      8   9.0  Summer   
1323       2026-08-21  2026-08-21 09:45:00  2026      8   9.0  Summer   
1324       2026-08-21  2026-08-21 07:43:00  2026      8   7.0  Summer   
1325       2026-08-20  2026-08-20 16:16:00  2026      8  16.0  Summer   

      has_observation_time  
1321                  True  
1322                  True  
1323                  True  
1324                  True  
1325                  True  


In [60]:
STATE_BOUNDS = {
    "CA": {
        "lat_min": 32.5,
        "lat_max": 42.0,
        "lon_min": -124.5,
        "lon_max": -114.1,
    },
    "AZ": {
        "lat_min": 31.3,
        "lat_max": 37.0,
        "lon_min": -114.9,
        "lon_max": -109.0,
    },
    "TX": {
        "lat_min": 25.8,
        "lat_max": 36.5,
        "lon_min": -106.7,
        "lon_max": -93.5,
    },
}

In [61]:
def coordinate_in_state(row):
    bounds = STATE_BOUNDS[row["state"]]

    return (
        bounds["lat_min"] <= row["latitude"] <= bounds["lat_max"]
        and
        bounds["lon_min"] <= row["longitude"] <= bounds["lon_max"]
    )


ebird_metadata["coordinate_valid"] = (
    ebird_metadata.apply(
        coordinate_in_state,
        axis=1
    )
)

print(
    ebird_metadata["coordinate_valid"]
    .value_counts(dropna=False)
)

coordinate_valid
True     1323
False       3
Name: count, dtype: int64


In [62]:
invalid_coordinates = ebird_metadata[
    ~ebird_metadata["coordinate_valid"]
][
    [
        "state",
        "species_code",
        "common_name",
        "scientific_name",
        "location_name",
        "latitude",
        "longitude",
        "observation_date",
        "submission_id",
    ]
]

print(
    invalid_coordinates.to_string(index=False)
)

state species_code              common_name         scientific_name                                                  location_name  latitude   longitude observation_date submission_id
   CA       tufpuf            Tufted Puffin     Fratercula cirrhata                        North Pacific Ocean (39.4248,-124.5988) 39.424843 -124.598837       2026-09-18    S394029370
   CA       ftspet Fork-tailed Storm-Petrel     Hydrobates furcatus                        North Pacific Ocean (40.2152,-124.9891) 40.215156 -124.989111       2026-09-18    S393988939
   CA       layalb         Laysan Albatross Phoebastria immutabilis North Wind Ornithology Pelagic -- Sep 13th 2026 -- Chum Stop 2 40.761093 -124.753948       2026-09-13    S392973314


In [63]:
ebird_metadata["coordinate_valid"] = (
    ebird_metadata["latitude"].between(-90, 90)
    & ebird_metadata["longitude"].between(-180, 180)
)

print(
    ebird_metadata["coordinate_valid"]
    .value_counts(dropna=False)
)

coordinate_valid
True    1326
Name: count, dtype: int64


In [64]:
ebird_metadata["state_bbox_match"] = (
    ebird_metadata.apply(
        coordinate_in_state,
        axis=1
    )
)

print(
    ebird_metadata["state_bbox_match"]
    .value_counts(dropna=False)
)

state_bbox_match
True     1323
False       3
Name: count, dtype: int64


In [65]:
ebird_metadata["geographic_note"] = pd.Series(
    pd.NA,
    index=ebird_metadata.index,
    dtype="string"
)

ebird_metadata.loc[
    ebird_metadata["state_bbox_match"] == False,
    "geographic_note"
] = "Pelagic observation outside simplified state bounding box"

In [66]:
print(
    ebird_metadata[
        ebird_metadata["state_bbox_match"] == False
    ][
        [
            "state",
            "common_name",
            "latitude",
            "longitude",
            "state_bbox_match",
            "geographic_note"
        ]
    ].to_string(index=False)
)

state              common_name  latitude   longitude  state_bbox_match                                           geographic_note
   CA            Tufted Puffin 39.424843 -124.598837             False Pelagic observation outside simplified state bounding box
   CA Fork-tailed Storm-Petrel 40.215156 -124.989111             False Pelagic observation outside simplified state bounding box
   CA         Laysan Albatross 40.761093 -124.753948             False Pelagic observation outside simplified state bounding box


In [67]:
print("Missing how_many:",
      ebird_metadata["how_many"].isna().sum())

print("Negative counts:",
      (ebird_metadata["how_many"] < 0).sum())

print("Zero counts:",
      (ebird_metadata["how_many"] == 0).sum())

print("\nSummary:")
print(
    ebird_metadata["how_many"].describe()
)

Missing how_many: 16
Negative counts: 0
Zero counts: 0

Summary:
count    1310.000000
mean        3.664885
std        20.325538
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       600.000000
Name: how_many, dtype: float64


In [68]:
missing_counts = ebird_metadata[
    ebird_metadata["how_many"].isna()
][
    [
        "state",
        "species_code",
        "common_name",
        "scientific_name",
        "location_name",
        "observation_date",
        "latitude",
        "longitude",
        "observation_valid",
        "observation_reviewed",
        "submission_id",
    ]
]

print(
    missing_counts.to_string(index=False)
)

state species_code                                 common_name                 scientific_name                                                                   location_name observation_date  latitude   longitude  observation_valid  observation_reviewed submission_id
   CA       pinjay                                  Pinyon Jay       Gymnorhinus cyanocephalus                           Mono Basin National Forest Scenic Area Visitor Center       2026-09-19 37.966780 -119.120333               True                 False    S394276735
   CA       compea                              Indian Peafowl                  Pavo cristatus                                   Los Angeles County Arboretum & Botanic Garden       2026-09-18 34.143880 -118.051864               True                 False    S394083570
   CA       x00958 Red-crowned x Lilac-crowned Amazon (hybrid) Amazona viridigenalis x finschi                                 Angels Community Park (Parrot Roost), Santa Ana       2026-09-10 3

In [69]:
print("Missing species codes:",
      ebird_metadata["species_code"].isna().sum())

print("Unique species codes:",
      ebird_metadata["species_code"].nunique())

print("Unique common names:",
      ebird_metadata["common_name"].nunique())

print("Unique scientific names:",
      ebird_metadata["scientific_name"].nunique())

Missing species codes: 0
Unique species codes: 675
Unique common names: 675
Unique scientific names: 675


In [70]:
code_to_scientific = (
    ebird_metadata
    .groupby("species_code")["scientific_name"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    "Species codes mapped to multiple scientific names:"
)

print(
    code_to_scientific[
        code_to_scientific > 1
    ]
)

Species codes mapped to multiple scientific names:
Series([], Name: scientific_name, dtype: int64)


In [71]:
code_to_common = (
    ebird_metadata
    .groupby("species_code")["common_name"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    "Species codes mapped to multiple common names:"
)

print(
    code_to_common[
        code_to_common > 1
    ]
)

Species codes mapped to multiple common names:
Series([], Name: common_name, dtype: int64)


## eBird Species Identifier Audit

The standardized eBird dataset contains 1,326 observations representing 675 unique species codes. No species codes, common names, or scientific names are missing. Each eBird species code maps consistently to a single common name and scientific name within the acquired dataset, and no conflicting species-label mappings were detected.

No species-label corrections were therefore required during preprocessing. The original eBird species code is retained as the source-specific species identifier, while the common and scientific names are preserved for interpretation and cross-source analysis.

In [72]:
print(
    "Exact duplicate rows:",
    ebird_metadata.duplicated().sum()
)

Exact duplicate rows: 0


In [73]:
print(
    "Duplicate submission IDs:",
    ebird_metadata["submission_id"].duplicated().sum()
)

Duplicate submission IDs: 526


In [74]:
duplicate_species_observations = (
    ebird_metadata.duplicated(
        subset=[
            "state",
            "species_code",
            "location_id",
            "observation_datetime",
            "submission_id"
        ]
    )
)

print(
    "Duplicate species-observation records:",
    duplicate_species_observations.sum()
)

Duplicate species-observation records: 0


In [75]:
ebird_metadata["flag_missing_count"] = (
    ebird_metadata["how_many"].isna()
)

ebird_metadata["flag_missing_time"] = (
    ~ebird_metadata["has_observation_time"]
)

ebird_metadata["flag_pelagic"] = (
    ~ebird_metadata["state_bbox_match"]
)

ebird_metadata["flag_any"] = (
    ebird_metadata[
        [
            "flag_missing_count",
            "flag_missing_time",
            "flag_pelagic"
        ]
    ]
    .any(axis=1)
)

ebird_metadata["cleaning_action"] = "KEEP"

ebird_metadata["cleaning_reason"] = "Valid eBird observation"

ebird_metadata.loc[
    ebird_metadata["flag_any"],
    "cleaning_reason"
] = "Retained with documented metadata limitation"

In [76]:
print("eBird cleaning summary")

print(
    "Total observations:",
    len(ebird_metadata)
)

print(
    "Flagged observations:",
    ebird_metadata["flag_any"].sum()
)

print(
    "\nMissing count:",
    ebird_metadata["flag_missing_count"].sum()
)

print(
    "Missing time:",
    ebird_metadata["flag_missing_time"].sum()
)

print(
    "Pelagic / outside state box:",
    ebird_metadata["flag_pelagic"].sum()
)

eBird cleaning summary
Total observations: 1326
Flagged observations: 20

Missing count: 16
Missing time: 1
Pelagic / outside state box: 3


## eBird Cleaning Summary

The eBird dataset contains 1,326 observations across California, Arizona, and Texas. All observations were retained because no records were identified as technically invalid or exact duplicates. Twenty observations were flagged for documented metadata limitations: 16 lacked a reported individual count, one lacked an observation time, and three California observations were pelagic records located outside the simplified terrestrial state bounding box.

Missing observation counts were retained as missing rather than imputed, because the absence of a reported count does not imply an abundance of zero or one. The three pelagic observations were also retained because their geographic coordinates are valid and represent legitimate offshore observations. These flags are preserved for downstream analysis and interpretation.

In [78]:
print("=== FINAL eBird VALIDATION ===")

print("Rows:", len(ebird_metadata))
print("Unique submission IDs:", ebird_metadata["submission_id"].nunique())
print("Unique species:", ebird_metadata["species_code"].nunique())

print("\nMissing species codes:",
      ebird_metadata["species_code"].isna().sum())

print("Missing dates:",
      ebird_metadata["observation_date"].isna().sum())

print("Missing times:",
      (~ebird_metadata["has_observation_time"]).sum())

print("Missing coordinates:",
      ebird_metadata[["latitude", "longitude"]].isna().any(axis=1).sum())

print("Invalid coordinates:",
      (~ebird_metadata["coordinate_valid"]).sum())

print("Exact duplicate rows:",
      ebird_metadata.duplicated().sum())

print("Duplicate species-observations:",
      duplicate_species_observations.sum())

print("Flagged observations:",
      ebird_metadata["flag_any"].sum())

print("\nRecords by state:")
print(ebird_metadata["state"].value_counts())

=== FINAL eBird VALIDATION ===
Rows: 1326
Unique submission IDs: 800
Unique species: 675

Missing species codes: 0
Missing dates: 0
Missing times: 1
Missing coordinates: 0
Invalid coordinates: 0
Exact duplicate rows: 0
Duplicate species-observations: 0
Flagged observations: 20

Records by state:
state
CA    495
TX    458
AZ    373
Name: count, dtype: Int64


In [79]:
master_output_path = PROCESSED_DIR / "master_metadata.csv"
master_metadata.to_csv(master_output_path, index=False)
print(f"Saved {len(master_metadata)} rows to {master_output_path}")

ebird_output_path = PROCESSED_DIR / "ebird_observations.csv"
ebird_metadata.to_csv(ebird_output_path, index=False)
print(f"Saved {len(ebird_metadata)} rows to {ebird_output_path}")

# Final combined summary — mirrors the audit notebook's summary table,
# useful for the DataPrep_EDA tab and for anyone auditing your pipeline
final_summary = pd.DataFrame({
    "Metric": ["Xeno-canto recordings", "Unique species (acoustic)", "Flagged recordings",
               "eBird observations", "Unique species (eBird)", "Flagged eBird observations"],
    "Value": [len(master_metadata), master_metadata["species_id"].nunique(), master_metadata["flag_any"].sum(),
              len(ebird_metadata), ebird_metadata["species_code"].nunique(), ebird_metadata["flag_any"].sum()]
})
final_summary.to_csv(PROCESSED_DIR / "cleaning_summary.csv", index=False)
print(final_summary.to_string(index=False))

Saved 275 rows to c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\master_metadata.csv
Saved 1326 rows to c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\ebird_observations.csv
                    Metric  Value
     Xeno-canto recordings    275
 Unique species (acoustic)    159
        Flagged recordings     67
        eBird observations   1326
    Unique species (eBird)    675
Flagged eBird observations     20


# Final Cleaning Handoff

The preprocessing stage produced two analysis-ready datasets while preserving the original source data and all documented quality-control decisions.

The Xeno-canto dataset contains 275 unique acoustic recordings representing 159 species. All recordings were technically decodable and were retained for downstream acoustic analysis. Sixty-seven recordings carry one or more quality-control flags related to short duration, low amplitude, or clipping. These recordings were not removed because potentially challenging acoustic observations are important for later robustness and signal-enhancement experiments.

The eBird dataset contains 1,326 observations representing 675 species. No exact duplicate observations or inconsistent species-code mappings were identified. Twenty observations contain documented metadata limitations: 16 lack a reported observation count, one lacks an observation time, and three are valid pelagic observations outside the simplified terrestrial state bounding box. These observations were retained rather than discarded.

The processed datasets preserve source metadata, derived temporal variables, geographic validation fields, and quality-control flags. The decoded audio duration is used as the authoritative duration for acoustic analysis, while source-reported duration is retained for provenance. The resulting datasets are ready for acoustic feature extraction, exploratory data analysis, dimensionality reduction, clustering, and subsequent machine-learning experiments.